# Burgers: conditioning of the Gram matrix

We learn the solution operator $K: u_0 \mapsto u(\cdot, T)$ of the viscous Burgers' equation
$u_t + u u_x = \nu u_{xx}$ on $(0,1)$, $u(0,t) = u(1,t) = 0$, with $T = 0.2$.

* Inputs and outputs are sine coefficients in the basis $\sqrt 2 \sin(n\pi x)$; data come from the spectral Galerkin solver in `burgers_solver.py`.
* Input measure: $d_{\mathrm{in}} = 20$ independent coefficients, the $n$-th distributed as $\mathrm{Jacobi}(\alpha_n, \alpha_n)$. **As run:** $\alpha_n = (n-1)^2$, $n = 1, \dots, 20$ (so the first mode is uniform on $[-1,1]$).
* Approximation space: polynomial operators $\psi_m\, p_\lambda(u_0)$ with $\lambda$ in an anisotropic index set with weights $w_j$ = `linspace(1, 0.01, 20)` (the weights enter as an exponent $1/w_j$, see `owls.index_set`) and univariate degree at most 10.

For hyperbolic cross sets of increasing size we draw $M = \lfloor N \log N \rfloor$ inputs from the optimal measure $\mu$ or from $\rho$ and record $\mathrm{cond}(\mathbf G)$ (3 repetitions). No PDE solves are needed.

**Status:** the data in `data/` are from the arXiv v1 run; this experiment was not rerun after the cleanup.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
HC = np.arange(10, 70, 10)   # hyperbolic cross parameters k
RUNS = 3
MAX_SIZE = 5000              # as run: this caps HC(60) at 5000 of its 5046 indices
SEED = 0
if QUICK:
    HC, RUNS = np.array([5, 10]), 2

D_IN = 20                                             # input sine modes
MAX_DEGREE = 10                                       # max univariate polynomial degree
measures = [owls.Jacobi(a**2) for a in range(D_IN)]  # as run: alpha_n = (n - 1)^2 for n = 1..20
weights = np.linspace(1, 0.01, D_IN)                 # anisotropy weights

In [ ]:
if RECOMPUTE or QUICK:
    dims = np.zeros(len(HC))
    cond = {True: np.zeros((len(HC), RUNS)), False: np.zeros((len(HC), RUNS))}
    for i, k in enumerate(HC):
        L = owls.index_set(D_IN, k, "hc", weights, MAX_DEGREE, MAX_SIZE)
        N = len(L)
        M = int(N * np.log(N))
        dims[i] = N
        for r in range(RUNS):
            for optimal in (True, False):
                rng = np.random.default_rng([SEED, i, r, int(optimal)])
                x = owls.sample_optimal(M, L, measures, rng) if optimal else owls.sample_rho(M, measures, rng)
                V = owls.vandermonde(x, L, measures)
                w = N / np.sum(V**2, axis=1) if optimal else np.ones(M)   # least-squares weights
                cond[optimal][i, r] = owls.gram_cond(np.sqrt(w / M)[:, None] * V)
        print(f"k = {k}: N_eff = {N}, mean cond(G) optimal {cond[True][i].mean():.2f}, Monte Carlo {cond[False][i].mean():.2e}")
    np.save(os.path.join(DATA, "Condition_Ind.npy"), cond[True])
    np.save(os.path.join(DATA, "Condition_Std.npy"), cond[False])
    np.save(os.path.join(DATA, "Condition_Dims.npy"), dims)

In [ ]:
C_ind = np.load(os.path.join(DATA, "Condition_Ind.npy"))
C_std = np.load(os.path.join(DATA, "Condition_Std.npy"))
dims = np.load(os.path.join(DATA, "Condition_Dims.npy"))

fig_width, fig_height = paper_style(width=0.5, aspect=0.6, fontsize=7, legend_fontsize=5)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
ax.set_yscale("log")
for vals, style, label in [(C_std, "kv--", r"Standard Sampling: $f^i\sim \rho$"), (C_ind, "r^--", r"Induced Sampling: $f^i\sim \mu$")]:
    for i, N in enumerate(dims):
        ax.plot([N, N], [vals[i].min(), vals[i].max()], "k--", linewidth=0.5)
        ax.plot(np.full(vals.shape[1], N), vals[i], "ko", markersize=1)
    ax.plot(dims, vals.mean(axis=1), style, markersize=3, linewidth=1, label=label)
ax.set_xlabel("\n" + r"$N_{\mathrm{eff}}(k; \mathrm{HC})$ for $ k\in \{$" + f"{HC[0]}, {HC[1]}, ..., {HC[-1]}" + r"$\}$")
ax.set_ylabel(r"$\mathrm{cond}(\mathbf{G})$")
ax.legend(loc="best")
savefig(os.path.join(FIGS, "vB_CondNum.pdf"))
plt.show()